# Output data integrity checks: tas, tasmax, tasmin

From issue #450
```
there are no nans in the output
no time slices are exactly the same
output variables are within broadly reasonable ranges (same as the input data checks in Add input data checks #316)
```

Four checks against the consolidated output store, one section each:

`
1. **No NaNs beyond the static edge cells**
2. **No duplicate time slices**
3. **Reasonable ranges** (`VAR_SPATIAL_RANGES` in `srm.qaqc`, same bounds as the input checks in #316)
4. **Within input time bounds** (`resolve_member_time_bounds`)


In [1]:
import dask
import pandas as pd
import xarray as xr

from srm.qaqc import VAR_SPATIAL_RANGES
from srm.validation import _open_output_datatree, resolve_member_time_bounds

In [2]:
import os

os.environ["FRISKY_SUMMARY"] = "off"

from dask_array.xarray import register
from distributed import Client
from frisky import hijack

register()
client = hijack(Client())
client

<frisky.Client: scheduler="127.0.0.1:38721" id="client-0">

In [3]:
STORE_URI = (
    "s3://carbonplan-scratch/srm/outputs/qa/CESM2-WACCM-ERA5-lat-35.0to-22.0_lon16.0to33.0.icechunk"
)
BRANCH = "main"
GCM = "CESM2-WACCM"
VARIABLES = ["tas", "tasmax", "tasmin"]
SPATIAL = ["lat", "lon"]

# Subset configs select slightly more area than the data covers, leaving static all-NaN edge
# cells. Set False for a global run, where check 1 then requires zero NaNs of any kind.
ALLOW_STATIC_NAN_CELLS = True

# On-disk group name -> canonical scenario name used by resolve_member_time_bounds.
GROUP_TO_SCENARIO = {
    "historical": "historical",
    "ssp245": "SSP245",
}  # "g6_1p5k": "G6-1.5K"}

tree = _open_output_datatree(STORE_URI, branch=BRANCH)

# One lazy DataArray per (scenario, variable, member). Members stay separate rows: within a
# scenario they can span different time ranges (some CESM2 SSP245 members end 2070, others
# 2100), so stacking them along an ensemble_member dim would pad NaNs and break check 1.
leaves = {
    (s, v, m): tree[f"{s}/{v}/{m}"].dataset[v]
    for s in GROUP_TO_SCENARIO
    for v in VARIABLES
    if v in tree[s].children
    for m in tree[s][v].children
}


def table(rows: dict) -> pd.DataFrame:
    """One row per leaf, indexed (scenario, variable, member)."""
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis(
        ["scenario", "variable", "member"]
    )


print(f"{len(leaves)} (scenario, variable, member) leaves to check")

4 (scenario, variable, member) leaves to check


In [4]:
tree

<xarray.DataTree>
Group: /
├── Group: /historical
│   ├── Group: /historical/dtr
│   │   └── Group: /historical/dtr/001
│   │           Dimensions:  (time: 13514, lat: 53, lon: 69)
│   │           Coordinates:
│   │             * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│   │             * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│   │             * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│   │           Data variables:
│   │               dtr      (time, lat, lon) float32 198MB dask.array<open_dataset-dtr, shap...
│   │           Attributes: (12/17)
│   │               Conventions:                                 CF-1.8
│   │               institution:                                 CarbonPlan
│   │               history:                                     2026-07-16T20:00:48Z: BCSD d...
│   │               srm_downscaling:version:                     0.9.2.post0+g753257644.d2026...
│   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │               srm_downscaling:scenario:                    SSP245
│   │               ...                                          ...
│   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │               srm_downscaling:downscaling_method:          multiplicative
│   │               srm_downscaling:train_period:                1978-2014
│   │               srm_downscaling:config_hash:                 172f48801757
│   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │               srm_downscaling:creation_date:               2026-07-16
│   ├── Group: /historical/tas
│   │   └── Group: /historical/tas/r2i1p1f1
│   │           Dimensions:  (time: 13514, lat: 53, lon: 69)
│   │           Coordinates:
│   │             * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│   │             * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│   │             * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│   │           Data variables:
│   │               tas      (time, lat, lon) float32 198MB dask.array<open_dataset-tas, shap...
│   │           Attributes: (12/17)
│   │               Conventions:                                 CF-1.8
│   │               institution:                                 CarbonPlan
│   │               history:                                     2026-07-16T20:00:44Z: BCSD d...
│   │               srm_downscaling:version:                     0.9.2.post0+g753257644.d2026...
│   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │               srm_downscaling:scenario:                    SSP245
│   │               ...                                          ...
│   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │               srm_downscaling:downscaling_method:          additive
│   │               srm_downscaling:train_period:                1978-2014
│   │               srm_downscaling:config_hash:                 7690315aba65
│   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │               srm_downscaling:creation_date:               2026-07-16
│   └── Group: /historical/tasmax
│       └── Group: /historical/tasmax/001
│               Dimensions:  (time: 13514, lat: 53, lon: 69)
│               Coordinates:
│                 * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│                 * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│                 * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│               Data variables:
│                   tasmax   (time, lat, lon) float32 198MB dask.array<open_dataset-tasmax, s...
│               Attributes:

In [5]:
tree

<xarray.DataTree>
Group: /
├── Group: /historical
│   ├── Group: /historical/dtr
│   │   └── Group: /historical/dtr/001
│   │           Dimensions:  (time: 13514, lat: 53, lon: 69)
│   │           Coordinates:
│   │             * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│   │             * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│   │             * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│   │           Data variables:
│   │               dtr      (time, lat, lon) float32 198MB dask.array<open_dataset-dtr, shap...
│   │           Attributes: (12/17)
│   │               Conventions:                                 CF-1.8
│   │               institution:                                 CarbonPlan
│   │               history:                                     2026-07-16T20:00:48Z: BCSD d...
│   │               srm_downscaling:version:                     0.9.2.post0+g753257644.d2026...
│   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │               srm_downscaling:scenario:                    SSP245
│   │               ...                                          ...
│   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │               srm_downscaling:downscaling_method:          multiplicative
│   │               srm_downscaling:train_period:                1978-2014
│   │               srm_downscaling:config_hash:                 172f48801757
│   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │               srm_downscaling:creation_date:               2026-07-16
│   ├── Group: /historical/tas
│   │   └── Group: /historical/tas/r2i1p1f1
│   │           Dimensions:  (time: 13514, lat: 53, lon: 69)
│   │           Coordinates:
│   │             * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│   │             * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│   │             * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│   │           Data variables:
│   │               tas      (time, lat, lon) float32 198MB dask.array<open_dataset-tas, shap...
│   │           Attributes: (12/17)
│   │               Conventions:                                 CF-1.8
│   │               institution:                                 CarbonPlan
│   │               history:                                     2026-07-16T20:00:44Z: BCSD d...
│   │               srm_downscaling:version:                     0.9.2.post0+g753257644.d2026...
│   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │               srm_downscaling:scenario:                    SSP245
│   │               ...                                          ...
│   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │               srm_downscaling:downscaling_method:          additive
│   │               srm_downscaling:train_period:                1978-2014
│   │               srm_downscaling:config_hash:                 7690315aba65
│   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │               srm_downscaling:creation_date:               2026-07-16
│   └── Group: /historical/tasmax
│       └── Group: /historical/tasmax/001
│               Dimensions:  (time: 13514, lat: 53, lon: 69)
│               Coordinates:
│                 * time     (time) datetime64[ns] 108kB 1978-01-01 1978-01-02 ... 2014-12-31
│                 * lat      (lat) float32 212B -35.0 -34.75 -34.5 -34.25 ... -22.5 -22.25 -22.0
│                 * lon      (lon) float32 276B 16.0 16.25 16.5 16.75 ... 32.25 32.5 32.75 33.0
│               Data variables:
│                   tasmax   (time, lat, lon) float32 198MB dask.array<open_dataset-tasmax, s...
│               Attributes:

## Compute all array statistics in one pass

Everything checks 1-3 need per leaf: NaN cell counts, global min/max, and per-day spatial
fingerprints (mean/std/min/max) for the duplicate check. Single batched `dask.compute`.

In [6]:
def leaf_stats(da: xr.DataArray) -> xr.Dataset:
    nan = da.isnull()
    static = nan.all("time")
    return xr.Dataset(
        {
            "n_static_cells": static.sum(),
            "n_partial_cells": (nan.any("time") & ~static).sum(),
            "min": da.min(),
            "max": da.max(),
            "fingerprint": xr.concat(
                [da.mean(SPATIAL), da.std(SPATIAL), da.min(SPATIAL), da.max(SPATIAL)],
                dim=pd.Index(["mean", "std", "min", "max"], name="stat"),
            ),
        }
    )


(stats,) = dask.compute({key: leaf_stats(da) for key, da in leaves.items()})

## Check 1 -- no NaNs beyond the static edge cells

The South Africa subset config selects slightly more area than the data covers, so a fixed
set of edge cells is NaN on *every* day (static). A cell NaN on some-but-not-all days
(partial) is a data bug -- the stale-tasmin signature on fix-v1/v2/v3.

Pass = zero partial-NaN cells; with `ALLOW_STATIC_NAN_CELLS = False` (global run) static
cells must also be zero, making this the literal no-NaNs check.

In [7]:
nan_df = table(
    {
        k: {"n_static_cells": int(s.n_static_cells), "n_partial_cells": int(s.n_partial_cells)}
        for k, s in stats.items()
    }
)
nan_df["pass"] = (nan_df.n_partial_cells == 0) & (
    ALLOW_STATIC_NAN_CELLS | (nan_df.n_static_cells == 0)
)
nan_df

n_static_cells  n_partial_cells  pass
scenario   variable member                                         
historical tas      r2i1p1f1             423                0  True
           tasmax   001                  423                0  True
ssp245     tas      007                  423                0  True
           tasmax   007                  423                0  True

## Check 2 -- no duplicate time slices

Two identical days necessarily share a fingerprint, so collisions are a complete candidate
set; each collision is confirmed elementwise (loads only the two candidate slices, and only
when a collision exists).

In [8]:
def duplicate_pairs(da: xr.DataArray, fp: xr.DataArray) -> list[tuple[str, str]]:
    """Confirmed identical-day pairs: fingerprint collision + exact elementwise equality."""
    df = fp.round(10).to_pandas().T.dropna()  # rows=time, cols=stat; all-NaN days drop out
    collided = df[df.duplicated(keep=False)].groupby(list(df.columns)).groups
    # drop the scalar time coord before comparing -- .equals compares coords too, and the
    # differing time stamps would otherwise mask genuinely identical data
    return [
        (str(times[0])[:10], str(t)[:10])
        for times in collided.values()
        for t in times[1:]
        if da.sel(time=t).drop_vars("time").equals(da.sel(time=times[0]).drop_vars("time"))
    ]


dup_df = table(
    {k: {"duplicate_pairs": duplicate_pairs(leaves[k], s.fingerprint)} for k, s in stats.items()}
)
dup_df["pass"] = dup_df.duplicate_pairs.str.len() == 0
dup_df

duplicate_pairs  pass
scenario   variable member                        
historical tas      r2i1p1f1              []  True
           tasmax   001                   []  True
ssp245     tas      007                   []  True
           tasmax   007                   []  True

## Check 3 -- reasonable ranges

Whole-array min/max per leaf against the `VAR_SPATIAL_RANGES` intervals.

In [9]:
range_df = table({k: {"min": float(s["min"]), "max": float(s["max"])} for k, s in stats.items()})
range_df["pass"] = [
    VAR_SPATIAL_RANGES[v]["min"][0] <= mn <= VAR_SPATIAL_RANGES[v]["min"][1]
    and VAR_SPATIAL_RANGES[v]["max"][0] <= mx <= VAR_SPATIAL_RANGES[v]["max"][1]
    for (s, v, m), mn, mx in zip(range_df.index, range_df["min"], range_df["max"])
]
range_df

min         max  pass
scenario   variable member                                
historical tas      r2i1p1f1  263.803558  309.892273  True
           tasmax   001       267.660309  318.025024  True
ssp245     tas      007       264.646057  313.625793  True
           tasmax   007       268.191986  322.397644  True

## Check 4 -- within input time bounds

Metadata only, no compute. G6/SAI outputs legitimately start before the raw scenario window --
the pipeline bridges 2015-2035 with SSP245 (`stitch_historical_scenario`), mirroring the
`is_sai_scenario` carve-out in `check_config_time_domain` -- so the start bound is enforced
only for non-SAI scenarios.

In [10]:
bounds_df = table(
    {
        (s, v, m): {
            "actual_start": str(da.time.values[0])[:10],
            "actual_end": str(da.time.values[-1])[:10],
            "expected": resolve_member_time_bounds(GCM, GROUP_TO_SCENARIO[s], m),
        }
        for (s, v, m), da in leaves.items()
    }
)
expected_start = bounds_df.expected.str[0].fillna("0000-01-01")  # no known bounds -> pass
expected_end = bounds_df.expected.str[1].fillna("9999-12-31")
is_sai = bounds_df.index.get_level_values("scenario").str.upper().str.contains("G6|SAI")
bounds_df["pass"] = (is_sai | (bounds_df.actual_start >= expected_start)) & (
    bounds_df.actual_end <= expected_end
)
bounds_df

actual_start  actual_end  \
scenario   variable member                              
historical tas      r2i1p1f1   1978-01-01  2014-12-31   
           tasmax   001        1978-01-01  2014-12-31   
ssp245     tas      007        2015-01-01  2070-12-31   
           tasmax   007        2015-01-01  2070-12-31   

                                              expected  pass  
scenario   variable member                                    
historical tas      r2i1p1f1  (1850-01-01, 2015-12-31)  True  
           tasmax   001       (1978-01-01, 2015-12-31)  True  
ssp245     tas      007       (2015-01-01, 2070-12-31)  True  
           tasmax   007       (2015-01-01, 2070-12-31)  True

## Summary

In [11]:
summary = pd.concat(
    {
        "no_partial_nans": nan_df["pass"],
        "no_duplicate_timesteps": dup_df["pass"],
        "reasonable_range": range_df["pass"],
        "within_input_time_bounds": bounds_df["pass"],
    },
    axis=1,
)
print(summary.to_string())
print(f"\nAll checks passed across {len(summary)} leaves: {bool(summary.all().all())}")

                              no_partial_nans  no_duplicate_timesteps  reasonable_range  within_input_time_bounds
scenario   variable member                                                                                       
historical tas      r2i1p1f1             True                    True              True                      True
           tasmax   001                  True                    True              True                      True
ssp245     tas      007                  True                    True              True                      True
           tasmax   007                  True                    True              True                      True

All checks passed across 4 leaves: True


In [13]:
client.shutdown()

NameError: name 'cluster' is not defined